In [1]:
!pip install -q transformers==4.41.2 librosa soundfile gradio numpy scipy

In [2]:
import numpy as np
import librosa
import gradio as gr
import time
import soundfile as sf
import torch

from transformers import pipeline

In [ ]:
# Emotion Detection
emotion_model = pipeline(
    "audio-classification",
    model="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"
)


# Sound Event Detection
sound_model = pipeline(
    "audio-classification",
    model="MIT/ast-finetuned-audioset-10-10-0.4593"
)

# Keyword Spotting
kws_model = pipeline(
    "audio-classification",
    model="superb/wav2vec2-base-superb-ks"
)

# Speech Recognition (ASR)
asr_model = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base"
)

In [4]:
HIGH_PRIORITY = [
    "help me", "save me", "please help", "i need help",
    "call the police", "call police", "call ambulance",
    "i am in danger", "i am being attacked",
    "someone help me", "please save me",
    "i am hurt", "i am bleeding"
]

MEDIUM_PRIORITY = [
    "stop", "leave me", "go away", "no no",
    "don't touch me", "let me go",
    "i am scared", "i am afraid",
    "please no"
]

CONTEXT_WORDS = [
    "accident", "crash", "fire",
    "gun", "knife", "blood",
    "injured", "hospital", "danger", "attack"
]

SAFE_CONTEXT = [
    "help with homework",
    "help me study",
    "help me code",
    "can you help me",
    "need help with project"
]

In [5]:
def keyword_engine(text):

    text = text.lower()
    score = 0

    for k in HIGH_PRIORITY:
        if k in text:
            score += 0.7

    for k in MEDIUM_PRIORITY:
        if k in text:
            score += 0.4

    for k in CONTEXT_WORDS:
        if k in text:
            score += 0.3

    for k in SAFE_CONTEXT:
        if k in text:
            score -= 0.8

    return score

In [6]:
audio_buffer = []
BUFFER_SIZE = 3
SAMPLE_RATE = 16000

In [20]:
def preprocess_audio(audio, sr):

    if audio is None or len(audio) == 0:
        return np.zeros(16000, dtype=np.float32)

    # stereo → mono
    if len(audio.shape) > 1:
        audio = np.mean(audio, axis=1)

    # convert to float32
    if audio.dtype != np.float32:
        audio = audio.astype(np.float32)

        # normalize int16 → float
        if np.max(np.abs(audio)) > 1:
            audio = audio / 32768.0

    # resample
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)

    return audio

In [16]:
def detect_emotion(audio):
    result = emotion_model({"array": audio, "sampling_rate": 16000})
    return result[0]["label"], result[0]["score"]

def detect_sound(audio):
    result = sound_model({"array": audio, "sampling_rate": 16000})

    for r in result:
        label = r["label"].lower()
        if any(k in label for k in ["scream", "cry", "explosion", "gunshot", "glass"]):
            return True

    return False

def transcribe(audio):
    result = asr_model({"array": audio, "sampling_rate": 16000})
    return result["text"]

In [9]:
def transcribe(audio):
    result = asr_model(audio)
    return result["text"]

In [25]:
import os

def save_evidence(audio):
    try:
        filename = f"/content/evidence_{int(time.time())}.wav"

        # ensure directory exists
        os.makedirs("/content", exist_ok=True)

        sf.write(filename, audio.astype(np.float32), SAMPLE_RATE)


        if os.path.exists(filename) and os.path.isfile(filename):
            return filename
        else:
            return None

    except Exception as e:
        print("Save error:", e)
        return None

In [32]:
def fusion_engine(audio):
    try:

        score = 0

        emotion, emo_score = detect_emotion(audio)
        sound_flag = detect_sound(audio)
        motion_flag = detect_motion()

        if any(k in emotion.lower() for k in ["fear", "angry", "sad"]):
            score += 0.3

        if sound_flag:
            score += 0.3

        if motion_flag:
            score += 0.4

        if score < 0.5:
            return "🟢 Safe", "", None, score

        text = transcribe(audio)

        keyword_score = keyword_engine(text)
        score += keyword_score

        if score < 0.7:
            return"🟡 Suspicious", text, None, score

        evidence = save_evidence(audio)

        if evidence and os.path.isfile(evidence):
          file_output = evidence
        else:
          file_output = None

        print("Evidence path:", evidence)
        return "🚨 EMERGENCY", text, safe_file(evidence), score

    except Exception as e:
        print("ERROR:", str(e))
        return "Error", "Check console", "", 0

In [29]:
import os

def safe_file(file_path):
    if file_path is None:
        return None

    if isinstance(file_path, str) and os.path.isfile(file_path):
        return file_path

    return None

In [33]:
def app(audio):

    try:
        if audio is None:
            return "No input", "", None, ""

        sr, data = audio
        data = preprocess_audio(data, sr)

        audio_buffer.append(data)
        if len(audio_buffer) > BUFFER_SIZE:
            audio_buffer.pop(0)

        full_audio = np.concatenate(audio_buffer)
        full_audio = full_audio[-16000 * 5:]

        status, text, file, score = fusion_engine(full_audio)

        return status, text, safe_file(file), f"{score:.2f}"

    except Exception as e:
        print("APP ERROR:", e)
        return "Error", "Check logs", None, ""

In [ ]:
gr.Interface(
    fn=app,
    inputs=gr.Audio(sources=["microphone"], type="numpy"),
    outputs=[
        gr.Text(label="Status"),
        gr.Text(label="Transcription"),
        gr.File(label="Evidence Recording", type="filepath"),
        gr.Text(label="Risk Score")
    ],
    title="🚨 Multi-Signal Emergency Detection System",
    description="""
This system detects emergencies using:
• Emotion (fear/panic)
• Sound (scream/crash)
• Keywords (help/stop)
• Motion (impact)

Only triggers when multiple signals confirm danger.
"""
).launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d67b12babf64560376.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
